# 02_04 On your own: what are Kittiwake's one-star reviewers complaining about?

**The brief.** Kittiwake Mobile's product team has 400 app-store reviews in `data/kittiwake_reviews.csv`
(`review_id`, `stars`, `text`) and two questions.

1. **What distinguishes each rating?** For each star rating from 1 to 5, the ten terms that most
   distinguish that rating's reviews **from the other ratings'**. Not the most frequent words in them:
   "the" is frequent everywhere. Terms may be single words or bigrams.
2. **Can we find reviews on a topic?** The three reviews most relevant to each of these four queries:
   `"dropped calls"`, `"roaming charges"`, `"friendly support"` and
   `"the phone company took too much money"`.

Save one JSON file, `out/review_insights.json`, shaped like this:

```json
{"distinctive": {"1": ["term", ...], "2": [...], "3": [...], "4": [...], "5": [...]},
 "search": {"dropped calls": ["R...", "R...", "R..."], "roaming charges": [...],
            "friendly support": [...], "the phone company took too much money": [...]}}
```

**Choices that are yours.** For question 1, any TF-IDF comparison: the mean weight of each term in a rating's
reviews against the rest, or each rating's reviews joined into one document and TF-IDF over the five. For
question 2, any of the three search methods from 02_03, or a **hybrid** that combines two of them, one method
per query if you like. The last query is the one that decides: it shares almost no word with the reviews it
should find.

The check does not care how, only whether the answer is right: one-star terms that read as complaints,
five-star terms that read as praise, and search results that are about the query. `check_on_your_own()` is
exactly the checkpoint's test. For embeddings, `review_vectors(model, texts)` from `warm_embeddings` returns
the vectors computed when your session started (the import and model load take about fifteen seconds).

Running this in Google Colab? This cell sets it up; in CourseLabs it does nothing.

In [ ]:
# Colab setup. In a CourseLabs session this cell does nothing.
import os, sys
if "google.colab" in sys.modules:
    import importlib, importlib.util, subprocess
    LAB, REPO = "lab-nlp-02-turning-words-into-numbers", "/content/nlp-course"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/fenago/nlp-course.git", REPO], check=True)
    os.chdir(f"{REPO}/{LAB}")
    if not os.path.exists("data"):
        os.symlink("../data", "data")
    os.makedirs("out", exist_ok=True)
    os.environ["NLPLAB_DATA"] = f"{REPO}/data"
    sys.path.insert(0, os.getcwd())
    PIP = {'bm25s': 'bm25s',
           'sentence_transformers': 'sentence-transformers',
           'sklearn': 'scikit-learn',
           'pandas': 'pandas',
           'numpy': 'numpy'}
    missing = [spec for mod, spec in PIP.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
        importlib.invalidate_caches()
    print(f"Ready: {LAB} and its data are in {os.getcwd()}; installed {len(missing)} package(s).")
elif not os.path.isdir("/opt/nlplab/data") and os.path.isdir("data"):
    # A downloaded copy on your own computer: the helpers read data/ from here.
    os.environ["NLPLAB_DATA"] = os.path.abspath("data")

In [ ]:
import json
import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore", message="IProgress not found")   # a progress-bar notice, not an error
import bm25s
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nlpcheck import check_on_your_own
from warm_embeddings import MODEL, review_vectors   # the review vectors 02_03 used

reviews = pd.read_csv("data/kittiwake_reviews.csv")
queries = ["dropped calls", "roaming charges", "friendly support", "the phone company took too much money"]
print(reviews["stars"].value_counts().sort_index())
reviews.head()

In [ ]:
# YOUR CODE HERE
distinctive = {"1": [], "2": [], "3": [], "4": [], "5": []}
search = {q: [] for q in queries}

In [ ]:
os.makedirs("out", exist_ok=True)
json.dump({"distinctive": distinctive, "search": search}, open("out/review_insights.json", "w"), indent=1)
check_on_your_own();

When the check passes, go back to the lab instructions for the sign-off and the checkpoint.